In [ ]:
from modeling_distillemb import BertModel, BertForSequenceClassification
from distill_emb import DistillEmbSmall, DistillEmb
from config import DistillModelConfig, DistillEmbConfig
import torch
from transformers import AutoTokenizer, RwkvConfig, RwkvModel, AutoModel
from tokenizer import CharTokenizer
from knn_classifier import KNNTextClassifier
from data_loader import load_sentiment, load_ner_dataset, load_pos_dataset
from data_loader import load_news_dataset
import pandas as pd
from retrieval import build_json_pairs, top1_accuracy
import os
from transformers import GPT2LMHeadModel

In [ ]:
num_input_chars=12

In [ ]:
tokenizer = CharTokenizer.from_pretrained(pretrained_directory="distil-emb-base")
distill_config = DistillEmbConfig.from_pretrained(pretrained_model_name_or_path="distil-emb-base")
distill_model = DistillEmb.from_pretrained(pretrained_model_name_or_path="distil-emb-base")

In [ ]:
distill_config

In [ ]:
# distill_config.distill_dropout = 0.25
config = DistillModelConfig(
    vocab_size=30522,
    hidden_size=512,
    num_hidden_layers=1,
    num_attention_heads=8,
    intermediate_size=3072,
    max_position_embeddings=1024,
    type_vocab_size=2,
    pad_token_id=0,
    position_embedding_type="absolute",
    use_cache=True,
    classifier_dropout=None,
    hidden_dropout_prob=0.5,
    embedding_type="distill",  # 'distilemb', 'fasttext'
    encoder_type='lstm', #'lstm'
    num_input_chars=num_input_chars,  # number of characters in each token
    char_vocab_size=tokenizer.char_vocab_size,
    distill_config=distill_config,
    distill_pretrained_model_name="distil-emb-base",
    is_decoder=False
)


In [ ]:
path = "downstream-data/sentiment.parquet"
df = pd.read_parquet(path)
if 'sent' in path:
    # remove 0th index
    df = df[df['text'] != 'tweet'].reset_index(drop=True)

In [ ]:
import re

def anonymize_and_normalize_text(text: str, lowercase: bool = True) -> str:
    """
    Preprocess text exactly as in AfriSenti[](https://arxiv.org/pdf/2302.08956):
    - Replace all @mentions with '@user'
    - Remove all URLs
    - Optionally lowercase (used for Nigerian languages in the paper)
    - Clean up whitespace
    
    Args:
        text (str): Raw input text
        lowercase (bool): Set to True for Nigerian Pidgin, Hausa, etc.; False for others
    
    Returns:
        str: Cleaned text
    """
    if not isinstance(text, str):
        return text
    
    # 1. Replace @mentions with @user
    text = re.sub(r'@[\w]+', '@user', text)
    
    # 2. Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)                    # http:// or https://
    text = re.sub(r'www\.\S+', '', text)                          # www.
    text = re.sub(r'\b\S+\.(com|org|net|edu|gov)\b', '', text)     # domain.com
    
    # 3. Optional lowercasing (used in AfriSenti for Nigerian languages)
    if lowercase:
        text = text.lower()
    
    # 4. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
df

In [ ]:
len(df['lang'].unique())

In [ ]:
lang_counts = df.groupby('split')['lang'].nunique()
for split, count in lang_counts.items():
    print(f"{split.capitalize()} split has {count} languages.")

In [ ]:
label2id = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}

df['label'] = df['label'].map(label2id).astype(int)

config.label2id = label2id
config.id2label = id2label

print(f"Converted labels to integers: {label2id}")

In [ ]:
num_labels = len(df['label'].unique())
config.num_labels = num_labels
model = BertForSequenceClassification(config)

In [ ]:
# for param in model.bert.embeddings.word_embeddings.parameters():
#     param.requires_grad = False

In [ ]:
labels = [x.item() for x in df['label'].unique()]
print(labels)
text_col = 'text'

In [ ]:
from datasets import Dataset, DatasetDict
import random
import string
df['text'] = df[text_col].apply(anonymize_and_normalize_text)

def add_gibberish_noise(text: str, min_tokens: int = 1, max_tokens: int = 3, min_length: int = 3, max_length: int = 8) -> str:
    """
    Insert random gibberish tokens into the provided text for augmentation.
    """
    if not isinstance(text, str) or not text.strip():
        return text

    base_tokens = text.split()
    gibberish_tokens = [
        "".join(random.choices(string.ascii_lowercase, k=random.randint(min_length, max_length)))
        for _ in range(random.randint(min_tokens, max_tokens))
    ]
    insert_idx = random.randint(0, len(base_tokens))
    augmented_tokens = base_tokens[:insert_idx] + gibberish_tokens + base_tokens[insert_idx:]
    return " ".join(augmented_tokens)

def build_augmented_dataset(dataframe: pd.DataFrame, samples_per_row: int = 1, separator: str = " "):
    sentiment_aliases = {
        "negative": ("negative", "neg", "0"),
        "neutral": ("neutral", "neu", "1"),
        "positive": ("positive", "pos", "2"),
    }

    def canonical_name(label_id: int) -> str:
        label_name = id2label[label_id].lower()
        for canonical, aliases in sentiment_aliases.items():
            if any(alias in label_name for alias in aliases):
                return canonical
        return label_name

    canonical_to_id = {canonical_name(lbl): lbl for lbl in dataframe["label"].unique()}

    def resolve_label(label_a: int, label_b: int) -> int:
        if 0 in (label_a, label_b):
            return 0
        if 1 in (label_a, label_b):
            return 1
        return 2

    augmented_rows = []
    for _, row in dataframe.iterrows():
        base_text, base_label = row["text"], row["label"]
        if random.random() < 0:
            base_text = add_gibberish_noise(base_text, min_tokens=1, max_tokens=2, min_length=3, max_length=6)
        augmented_rows.append({"text": base_text, "label": base_label})

        if samples_per_row < 1:
            continue
        
        sampled = dataframe.sample(n=5, replace=True)
        for _, sampled_row in sampled.iterrows():
            combined_text = f"{base_text}{separator}{sampled_row['text']}"
            if random.random() < 0.2:
                combined_text = add_gibberish_noise(combined_text, min_tokens=1, max_tokens=4, min_length=3, max_length=10)
            if base_label != sampled_row["label"] and (1 not in [base_label, sampled_row["label"]]):
                continue
            combined_label = resolve_label(base_label, sampled_row["label"])
            augmented_rows.append({"text": combined_text, "label": combined_label})

    augmented_df = pd.DataFrame(augmented_rows)
    # remove duplicates on text
    augmented_df = augmented_df.drop_duplicates(subset=['text']).reset_index(drop=True)
    min_count = augmented_df["label"].value_counts().min()
    augmented_df = (
        augmented_df.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(n=min_count, random_state=42))
        .reset_index(drop=True)
    )
    return augmented_df

# Assuming df is your dataframe
# Split the data based on the 'split' column
train_df = df[df['split'] == 'train'][['text', 'label']]
test_df = df[df['split'] == 'test'][['text', 'label']]

# aug_train_df = build_augmented_dataset(train_df, samples_per_row=2, separator=" ")


In [ ]:
# aug_train_df

In [ ]:
# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset

In [ ]:
from typing import Dict, Any

def preprocess_function(examples: Dict[str, Any]):
    batch = tokenizer(
        examples["text"],
        padding=False,
        max_length=512,
        return_attention_mask=False,
    )

    batch["labels"] = examples["label"]
    return batch



tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

In [ ]:
len(train_dataset[0]['text'].split())

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding="longest",
            max_length=512,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        batch['labels'] = torch.tensor([f['labels'] for f in features], dtype=torch.long)
        return batch

data_collator = CustomDataCollator(tokenizer)

In [ ]:
##### from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted', labels=labels)
    f1_macro = f1_score(labels, predictions, average='macro', labels=labels)
    f1_micro = f1_score(labels, predictions, average='micro', labels=labels)
    return {"accuracy": acc, "f1_weighted": f1, "f1_macro": f1_macro, "f1_micro": f1_micro}


import os
dataloader_num_workers=os.cpu_count() - 1
batch_size = 16

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    weight_decay=0.1,
    report_to=[],
    eval_strategy="epoch",  
    save_total_limit=1,
    save_only_model=True,
    logging_strategy="steps",
    logging_steps=10,
    label_smoothing_factor=0.15,
    max_grad_norm=5.0,
    warmup_ratio=0.0,
    lr_scheduler_type="cosine",
    dataloader_num_workers=8,        # Number of CPU workers for data loading
    dataloader_pin_memory=True,      # Faster GPU transfer
    gradient_accumulation_steps=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model after training
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
# model = BertForSequenceClassification.from_pretrained("distil-emb-seqcls-lstm").cuda()
model = trainer.model
model.eval()
# Ensure 'language' column exists in df
test_df = df[df['split'] == 'test'][['text', 'label', 'lang']]
languages = test_df['lang'].unique()
per_language_f1 = {}

batch_size = 16

for lang in languages:
    if lang == 'tg' or lang == 'or':
        continue
    lang_df = test_df[test_df['lang'] == lang]
    texts = lang_df['text'].tolist()
    labels = lang_df['label'].values
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]
        tokenized = tokenizer(
            batch_texts,
            padding='longest',
            truncation=True,
            max_length=512,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        with torch.no_grad():
            inputs = {k: v.cuda() for k, v in tokenized.items()}
            outputs = model(**inputs)
            batch_preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            preds.extend(batch_preds)
    f1 = f1_score(labels, preds, average='macro', labels=labels)
    per_language_f1[lang] = f1

# Print per-language F1
for lang, f1 in per_language_f1.items():
    print(f"Language: {lang}, F1: {f1:.4f}")

# Average F1
average_f1 = sum(per_language_f1.values()) / len(per_language_f1)
print(f"Average F1 across languages: {average_f1:.4f}")

special_languages = ["tg", "or"]
special_f1_scores = {}

for lang in special_languages:
    lang_df = test_df[test_df["lang"] == lang]
    if lang_df.empty:
        print(f"No samples found for language '{lang}'.")
        continue

    lang_texts = lang_df["text"].tolist()
    lang_labels = lang_df["label"].values
    lang_preds = []

    for i in range(0, len(lang_texts), batch_size):
        batch_texts = lang_texts[i:i + batch_size]
        tokenized = tokenizer(
            batch_texts,
            padding="longest",
            truncation=True,
            max_length=512,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        with torch.no_grad():
            inputs = {k: v.cuda() for k, v in tokenized.items()}
            outputs = model(**inputs)
            batch_preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            lang_preds.extend(batch_preds)

    f1 = f1_score(lang_labels, lang_preds, average="macro", labels=lang_labels)
    per_language_f1[lang] = f1
    special_f1_scores[lang] = f1
    print(f"Language: {lang}, F1: {f1:.4f}")

if special_f1_scores:
    special_average_f1 = sum(special_f1_scores.values()) / len(special_f1_scores)
    print(f"Average F1 for special languages: {special_average_f1:.4f}")
else:
    print("No F1 scores computed for the requested languages.")

In [ ]:
# model.save_pretrained("distil-emb-news-lstm-best-256")

In [ ]:
# Filter the test data to exclude 'or' and 'tg' languages
combined_test_df = test_df[~test_df['lang'].isin(['or', 'tg'])]

# Extract texts and labels
texts = combined_test_df['text'].tolist()
labels = combined_test_df['label'].values
all_preds = []

# Set model to evaluation mode
model.eval()

# Perform inference in batches
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    tokenized = tokenizer(
        batch_texts,
        padding='longest',
        truncation=True,
        max_length=512,
        return_tensors="pt",
        return_attention_mask=True,
        padding_side="right"
    )
    with torch.no_grad():
        inputs = {k: v.cuda() for k, v in tokenized.items()}
        outputs = model(**inputs)
        batch_preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(batch_preds)

# Compute and print the F1 score for the combined dataset
combined_f1 = f1_score(labels, all_preds, average='macro')
print(f"F1 score on all test data (excluding 'or' and 'tg'): {combined_f1:.4f}")